# C13_E02 — STR con identificación recursiva (RLS)

**Capítulo:** 13 — Control adaptativo, autoajuste e identificación online  
**Identificador:** `C13_E02`  
**Objetivo pedagógico:** Implementar identificación online RLS con factor de olvido y demostrar convergencia.  
**Librerías:** numpy, matplotlib

## Ejemplo industrial

Self-tuning regulator sobre proceso con cambio paramétrico lento.

---

> **Aviso de uso responsable.** Este notebook es didáctico. No es código de producción. Cualquier implementación real requiere validación independiente. Ver `docs/politica_uso_responsable.md`.

## 1. RLS para identificación online de proceso ARX(1,1)

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import os
np.random.seed(0)

class RLS:
    """Recursive Least Squares con factor de olvido."""
    def __init__(self, n_params, lam=0.98, P0=1000.0):
        self.theta = np.zeros(n_params)
        self.P = np.eye(n_params)*P0
        self.lam = lam
    def update(self, phi, y):
        phi = np.asarray(phi, dtype=float)
        den = self.lam + phi @ self.P @ phi
        K = self.P @ phi / den
        e = y - phi @ self.theta
        self.theta = self.theta + K*e
        self.P = (self.P - np.outer(K, phi @ self.P)) / self.lam
        return self.theta.copy()

# Proceso real: y[k+1] = a*y[k] + b*u[k] (a y b cambian a mitad de simulación)
N = 200; t = np.arange(N)
u = 0.5*np.sin(0.1*t) + 0.3*np.sign(np.sin(0.05*t))    # excitación persistente
y = np.zeros(N)
a_true, b_true = 0.9, 0.1
estimados = []
rls = RLS(n_params=2, lam=0.95)
for k in range(1, N):
    if k == N//2:
        a_true, b_true = 0.7, 0.2     # cambio paramétrico
    y[k] = a_true*y[k-1] + b_true*u[k-1] + 0.01*np.random.randn()
    phi = [y[k-1], u[k-1]]
    estimados.append(rls.update(phi, y[k]))
estimados = np.array(estimados)

## 2. Visualización de la convergencia

In [2]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(estimados[:,0], label="â (estimado)")
ax.plot(estimados[:,1], label="b̂ (estimado)")
ax.axhline(0.9, ls='--', color='C0', alpha=0.4, label="a verdadero (antes)")
ax.axhline(0.7, ls=':',  color='C0', alpha=0.6, label="a verdadero (después)")
ax.axhline(0.1, ls='--', color='C1', alpha=0.4, label="b verdadero (antes)")
ax.axhline(0.2, ls=':',  color='C1', alpha=0.6, label="b verdadero (después)")
ax.axvline(N//2-1, color='red', alpha=0.3)
ax.set_xlabel("k"); ax.set_ylabel("parámetros estimados")
ax.legend(loc='best', fontsize=8); ax.grid(alpha=0.3)
ax.set_title("C13_E02 — RLS con factor de olvido ante cambio paramétrico")
out = '../../figures/cap13/fig_C13_F02_rls.png'
os.makedirs(os.path.dirname(out), exist_ok=True)
fig.tight_layout(); fig.savefig(out, dpi=120); plt.show()

## 3. Interpretación

El RLS estima `a` y `b` en línea. Antes del cambio (k < 100) los estimados convergen a los valores verdaderos. Tras el cambio (k = 100), el factor de olvido `λ = 0.95` permite que los nuevos valores reemplacen a los viejos en ≈ 1/(1−λ) = 20 muestras. **Riesgo:** sin excitación persistente (u constante), la covarianza P diverge y la identificación deja de ser fiable. La excitación senoidal + cuadrada usada aquí cumple el requisito.